In [1]:
import strahler
import os
os.environ["STRAHLER_PM_BACKEND"] = "simd"
from importlib.metadata import version as package_version
if package_version("strahler") != strahler.version():
    raise RuntimeError("Package/library version mismatch. Reinstall strahler and restart the kernel.")
print("strahler package and library versions match")

from IPython.display import HTML, display
from notebook_support import HERE, table
from lexicographic_nester import BACKENDS, host_svg
from family_benchmarks import (ORDERS, recorded, result_text, fixed_maps, shape_html)


strahler package and library versions match


# CABP_SWP: grow the protocol data size

We start from the CABP_SWP parity benchmark in the
[SYNTCOMP 2026 Keiren selection](https://github.com/SYNTCOMP/benchmarks/tree/4105caf1f1e5fd3b76657879bfce8021d130cbde/parity/keiren).
It traces back to Keiren's protocol equivalence-checking family. Its checked
parameter values are **data size 2 and 4**, with capacity and window size both 1.
Branching bisimulation and strong bisimulation are separate series, so changes
in the equivalence relation do not masquerade as parameter scaling.

The branching data-size-2 conversion is byte-identical to this repository's
`examples/cabp_swp.ehoa.pg`. Both strong-bisimulation cases come from the tagged v2026 repository
corpus as related comparisons. Only the branching data-size-2 anchor belongs
to the original 484-game release selection; the pinned
[release receipt](https://github.com/SYNTCOMP/benchmarks/blob/f3374c9cab0d829498b4c055ed00ac2375e88e67/selection/releases/syntcomp26v2/parity-v2026.csv)
confirms its filename and SHA-256. The larger branching case comes from
[Keiren's CC0 original archive](https://doi.org/10.6084/m9.figshare.6004130.v1),
converted through the pinned official `pg2hoa.py` and Knor pipeline. Its roughly
130 MB eHOA file exceeds the release's documented 50 MB selection cutoff.
It is a reconstructed extension, not an additional file shipped in that release.
The branching-simulation alias duplicates the selected branching-bisimulation
file, so we count it once.

All four measurements use the same split-vertex Knor representation. Their raw
priorities {0,3,4} normalize to {0,1,2}; the data parameter changes the graph,
while the priority and host-height bounds stay fixed.

In [2]:
info, recording = recorded("cabp")
games = info["games"]
table(["case", "parameter", "vertices", "edges", "AD h E/O", "STRPM h E/O", "AD k cap", "STRPM k cap", "t cap"],
      [[g["label"], g["parameter"], g["vertices"], g["edges"], g["AD_height"],
        [d + 1 for d in g["PM_depth"]], g["AD_k_max"], g["PM_k_max"], g["t_max"]]
       for g in games])

case,parameter,vertices,edges,AD h E/O,STRPM h E/O,AD k cap,STRPM k cap,t cap
"branching-bisim, data=2",2,875281,1469041,"[2, 2]","[2, 3]","[2, 2]","[2, 3]",19
"strong-bisim, data=2",2,125084,202028,"[2, 2]","[2, 3]","[2, 2]","[2, 3]",16
"branching-bisim, data=4",4,2512705,4230209,"[2, 2]","[2, 3]","[2, 2]","[2, 3]",21
"strong-bisim, data=4",4,339444,551252,"[2, 2]","[2, 3]","[2, 2]","[2, 3]",18


## Small hosts at the same requested pair

A host is $U^k_{t,h}$: k bounds Strahler number, t gives the leaf allowance,
and h counts levels including the root. These complete candidate hosts all use
**(k,t)=(2,3)**, a pair inside the fixed study below. AD has height 2 for both
players; STRPM has heights 2 and 3. The drawings show available positions,
rather than the occupied answer tree.

In [3]:
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:16px">' +
             host_svg(2, 3, 2, "AD E/O and STRPM Even: h=2") +
             host_svg(2, 3, 3, "STRPM Odd: h=3") + '</div>'))

## A recorded answer, without millions of vertex IDs

This is the occupied shape of the first game's independent Zielonka answer.
Node labels show the normalized priority bound and the numbers of vertices in
H, T and S. Those are certificate sets; the host height h above counts node
levels. The full winner bitsets, bound to the fixture checksum, accompany the
recording. A reference limit leaves this view unavailable and is reported separately.

In [4]:
first_reference = next(r for r in recording["references"] if r["game"] == games[0]["id"])
table(["reference case", "outcome", "seconds", "stage", "won E/O"],
      [[r["game"], r["outcome"], f"{r['seconds']:.2f}", r["stage"], r.get("won", "unknown")]
       for r in recording["references"]])
display(HTML(shape_html(first_reference.get("reference_shape"))))

reference case,outcome,seconds,stage,won E/O
cabp-branching-bisim-data2,solved,3.42,JSON reduction and winner validation,"[0, 875281]"
cabp-strong-bisim-data2,solved,0.58,JSON reduction and winner validation,"[0, 125084]"
cabp-branching-bisim-data4,solved,12.39,JSON reduction and winner validation,"[0, 2512705]"
cabp-strong-bisim-data4,solved,1.46,JSON reduction and winner validation,"[0, 339444]"


## Three adaptive orders: a 60-second first pass

We compare AD, symmetric AD and **Strahler PM**, each with and without iterative
shaving. Preprocessing and optional same-parity compression are off. STRPM
requests the full SIMD kernel; dispatch records identify any scalar fallback.
SSPM is available separately and has a bit budget rather than these host parameters.

Lexicographic order exhausts t before increasing k. Ratio uses Oink's integer
ratio heap and starts at (1,1). Table order tries the exact small-host size
order for the larger initial player height, then the ratio fallback. See
[the size analysis](HOST-ORDER.md) for the ordering and its height sensitivity.
Both players share the cursor while retaining their individual k clamps.

Every process has **60 seconds and 2 GiB**, with at most two benchmark workers.
References use separate processes and identical limits. The displayed time
covers input, native solving and verification, JSON output, streaming reduction,
and winner checks. Native-command time is recorded separately and includes its
JSON output. These end-to-end times should not be read as kernel speedups or
compared directly to notebook 08's older timing pipeline. This was a shared-machine
first pass: the recording notes the overlap of the adaptive CABP runs with the
last Nester jobs. Fixed maps and ladder runs followed after those jobs finished.

A cell gives the final visited (k,t) and seconds. With +iter, earlier rounds
contribute wins, so this final pair is not evidence that a fixed run at that
pair suffices. A timeout leaves sufficiency unknown. After a method/order/series
hits its first limit, larger parameters are marked beyond its measured frontier;
other methods, orders and series continue independently.

In [5]:
by_case = {(r["game"], r["backend"], r["order"]): r for r in recording["rows"]}
for order in ORDERS:
    print(order)
    table(["case", "vertices", *BACKENDS],
          [[g["label"], g["vertices"],
            *[result_text(by_case[g["id"], b, order]) for b in BACKENDS]] for g in games])

lexicographic


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
"branching-bisim, data=2",875281,timeout (60 s cap),"(1, 0), 41.96 s",timeout (60 s cap),"(1, 0), 39.88 s",timeout (60 s cap),"(2, 3), 48.13 s"
"strong-bisim, data=2",125084,"(1, 0), 0.95 s","(1, 0), 0.96 s","(1, 0), 0.72 s","(1, 0), 0.76 s","(2, 5), 3.28 s","(2, 2), 2.83 s"
"branching-bisim, data=4",2512705,beyond measured frontier,timeout (60 s cap),beyond measured frontier,timeout (60 s cap),beyond measured frontier,timeout (60 s cap)
"strong-bisim, data=4",339444,"(1, 0), 2.55 s","(1, 0), 2.59 s","(1, 0), 1.83 s","(1, 0), 1.89 s","(2, 5), 9.79 s","(2, 2), 9.37 s"


ratio


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
"branching-bisim, data=2",875281,timeout (60 s cap),"(1, 1), 41.58 s","(2, 3), 30.23 s","(1, 1), 32.56 s","(3, 7), 47.64 s","(2, 3), 26.23 s"
"strong-bisim, data=2",125084,"(1, 1), 1.21 s","(1, 1), 0.97 s","(1, 1), 0.72 s","(1, 1), 0.71 s","(3, 5), 1.88 s","(2, 2), 0.93 s"
"branching-bisim, data=4",2512705,beyond measured frontier,timeout (60 s cap),timeout (60 s cap),timeout (60 s cap),timeout (60 s cap),timeout (60 s cap)
"strong-bisim, data=4",339444,"(1, 1), 2.57 s","(1, 1), 2.62 s","(1, 1), 1.79 s","(1, 1), 1.86 s","(3, 5), 5.84 s","(2, 2), 2.34 s"


table


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
"branching-bisim, data=2",875281,timeout (60 s cap),"(1, 0), 41.75 s",timeout (60 s cap),"(1, 0), 37.72 s",timeout (60 s cap),"(2, 3), 35.71 s"
"strong-bisim, data=2",125084,"(1, 0), 0.97 s","(1, 0), 0.97 s","(1, 0), 0.72 s","(1, 0), 0.73 s","(2, 5), 3.13 s","(3, 2), 1.81 s"
"branching-bisim, data=4",2512705,beyond measured frontier,timeout (60 s cap),beyond measured frontier,timeout (60 s cap),beyond measured frontier,timeout (60 s cap)
"strong-bisim, data=4",339444,"(1, 0), 2.57 s","(1, 0), 2.76 s","(1, 0), 1.85 s","(1, 0), 1.89 s","(2, 5), 9.01 s","(3, 2), 5.39 s"


## What the first pass established

Count completed cases before comparing times: a fast timeout is not a solve.
The dispatch counts below cover returned PM results; a killed process cannot
supply a complete dispatch summary. Unary k=1 hosts use the scalar fallback.

In [6]:
table(["order", "method", "solved", "limited", "beyond frontier", "SIMD calls", "scalar calls"],
      [[order, backend,
        sum(r["outcome"] == "solved" for r in recording["rows"] if r["order"] == order and r["backend"] == backend),
        sum(r["outcome"] in ("timeout", "memory") for r in recording["rows"] if r["order"] == order and r["backend"] == backend),
        sum(r["outcome"] == "not measured" for r in recording["rows"] if r["order"] == order and r["backend"] == backend),
        *[sum((r.get("dispatch") or {}).get(name, 0) for r in recording["rows"]
              if r["order"] == order and r["backend"] == backend) if backend.startswith("PM") else "—"
          for name in ("simd", "scalar")]]
       for order in ORDERS for backend in BACKENDS])

order,method,solved,limited,beyond frontier,SIMD calls,scalar calls
lexicographic,AD,2,1,1,—,—
lexicographic,AD+iter,3,1,0,—,—
lexicographic,AD-sym,2,1,1,—,—
lexicographic,AD-sym+iter,3,1,0,—,—
lexicographic,PM,2,1,1,24,72
lexicographic,PM+iter,3,1,0,46,112
ratio,AD,2,1,1,—,—
ratio,AD+iter,3,1,0,—,—
ratio,AD-sym,3,1,0,—,—
ratio,AD-sym+iter,3,1,0,—,—


## Which fixed hosts suffice at data size 2?

We map both data-size-2 relations, every admissible k (**1…2 for AD and 1…3
for STRPM**), and **t=0…3**, with 60 seconds and 2 GiB per fixed run.
This keeps the anchor comparisons separate from the data-size-4 adaptive
scaling experiment. PM+iter explicitly repeats its fixed pair until complete
or stalled; it does not inherit wins from a search over different pairs.

Green means a measured solve; red means a stabilized partial result. Amber
marks an unknown result at a resource limit. A star confirms a coordinatewise
Pareto minimum because every strict predecessor was measured partial. A
diamond has open predecessors. Gray cells were not run because an observed
solve already prevents them being new minima. Other unmeasured cells remain
unknown. We infer no host inclusion when k changes.

In [7]:
maps, minima = fixed_maps(info, recording)
display(HTML(maps))
table(["case", "method", "confirmed minimal (k,t)", "observed nondominated", "time/memory limits"], minima)

case,method,"confirmed minimal (k,t)",observed nondominated,time/memory limits
"branching-bisim, data=2",AD-sym,"[(2, 3)]","[(2, 3)]",[]
"branching-bisim, data=2",AD-sym+iter,"[(1, 0)]","[(1, 0)]",[]
"branching-bisim, data=2",PM,[],[],[]
"branching-bisim, data=2",PM+iter,"[(2, 3)]","[(2, 3)]",[]
"strong-bisim, data=2",AD-sym,"[(1, 0)]","[(1, 0)]",[]
"strong-bisim, data=2",AD-sym+iter,"[(1, 0)]","[(1, 0)]",[]
"strong-bisim, data=2",PM,[],[],[]
"strong-bisim, data=2",PM+iter,"[(2, 2)]","[(2, 2)]",[]


## Two targeted plain-PM follow-ups

Plain PM returned partial answers throughout both declared t≤3 maps. We then
repeated its fastest successful adaptive pair as a fresh fixed run: **(3,7)**
for branching bisimulation and **(3,5)** for strong bisimulation. Both solved.
These are separate follow-ups, not extensions silently folded into the original
grid. Their unmeasured predecessors prevent a claim of Pareto minimality.

In [8]:
table(["case", "method", "fixed (k,t)", "outcome", "seconds", "independent winner check"],
      [[r["game"], r["backend"], (r["k"], r["t"]), r["outcome"],
        f"{r['seconds']:.2f}", r["independent_winner_check"]]
       for r in recording["followup_rows"]])

case,method,"fixed (k,t)",outcome,seconds,independent winner check
cabp-branching-bisim-data2,PM,"(3, 7)",solved,13.05,True
cabp-strong-bisim-data2,PM,"(3, 5)",solved,0.77,True


On the branching data-size-2 anchor, **ratio solves five of six methods**
within the first-pass budget; lexicographic and table order each solve three.
The larger branching instance has no measured success at 60 seconds; some
method/order combinations were beyond their anchor's measured frontier.
All methods and orders solve both strong-bisimulation sizes.

The fixed maps explain another difference. With shaving, AD-sym already
suffices at (1,0) on both anchors. PM+iter has confirmed minima (2,3) for
branching and (2,2) for strong bisimulation. Plain PM needs a larger t than
this first grid tested, and the separate fixed follow-ups witness solutions.

Exact size order helps us choose small hosts, but this pass gives no universal
runtime advantage to table order. Which search finishes and which individual
host suffices are different questions. The two published data values also do
not establish a growth law.

Sources and exact conversion provenance are in [cabp/README.md](cabp/README.md)
and [cabp/manifest.json](cabp/manifest.json).

## Reproduce or extend the measurements

Normal notebook execution checks the bundled fixtures and reads the saved data.
It does not repeat the long benchmark or fetch any inputs. `manifest.json` pins
sources, graph sizes and conversion checksums; the recording also pins the
worker, native executables, source commit, limits and winner references.

From a checkout, install the development requirements, build the native tools,
and run the shared collector. It checkpoints after each case and resumes only
when the worker, inputs, executables and limits match. Use a new output file for
a different budget. A solver crash or verification failure stops collection;
it is not classified as a benchmark frontier.

In [9]:
print("python scripts/gen_notebook_families.py --family cabp --check")
print("python jupyter-examples/family_benchmarks.py run --family cabp --build-dir build --seconds 60 --jobs 2")
print("Recorded build:", recording["version"])
print("Input hashes and recorded winner checks passed.")

python scripts/gen_notebook_families.py --family cabp --check
python jupyter-examples/family_benchmarks.py run --family cabp --build-dir build --seconds 60 --jobs 2
Recorded build: pg2adot 0.4.2.dev42+ge085906 (git v0.4.1-42-ge085906)
Input hashes and recorded winner checks passed.
